# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds the feature vector and then attacks it: label-derived columns, same-window summaries, and client memorization. The feature build and the "train-with / train-without" leak test were moved here from `w03_data_contract.ipynb` (ML-04) and strengthened with a timeline check, an honest feature set, and a grouped split.

In [1]:
import os
HF_TOKEN = os.getenv('hf_key')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

- One row = one **content item** (client × content), aggregated over a 60-day window ending 2025-08-31.
- **Features:** `impressions_prev30d` (days −60…−31), `impressions_last30d`, `avg_position_30d`, `clicks_30d` (days −30…0), plus static attributes `search_volume`, `main_intent`, `content_type`.
- **Label:** `ctr_label = clicks_30d / impressions_last30d` over the last 30 days.
- Only content with ≥ 70 impressions in the *previous* window survives, so predictions rest on enough history: **15,310 rows**.
- Everything below is aggregated in one query, so the window boundaries are all in one place and every column can be checked against the label window (Section 3).

In [ ]:
data_features = con.sql(f"""
WITH bounds AS (
    SELECT DATE '2025-08-31' AS end_d
),

windowed AS (
    SELECT
        q.client_hash_id,
        q.content_hash_id,

        ANY_VALUE(d.main_intent) AS main_intent,
        ANY_VALUE(d.content_type) AS content_type,
        ANY_VALUE(d.search_volume) AS search_volume,

        -- Previous 30 days (days -60 to -31)
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,

        -- Last 30 days (days -30 to end_d)
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks
                ELSE 0 END) AS clicks_30d,

        AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END
        ) AS avg_position_30d,

        100.0 *
        COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
            NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label

    FROM {TABLES['fact_daily']} q
    CROSS JOIN bounds b
    JOIN {TABLES['dim_content']} d
      ON q.content_hash_id = d.content_hash_id

    WHERE q.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY q.client_hash_id, q.content_hash_id

    HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                 AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) >= 70
)

SELECT *
FROM windowed
""").df()

print(f"{len(data_features):,} content items with enough history")
data_features.head()

### Why these features

- **impressions_prev30d** — historical measurement, known before the label window opens.
- **avg_position_30d** — observed search position; caution: it is measured over the *same* days as the label (see the hunt).
- **impressions_last30d** — historical impressions; caution: it is the label's denominator (same window).
- **search_volume** — external keyword demand, independent of page performance.
- **main_intent** — inferred from the query before any optimization decision.
- **content_type** — decided when the page is created.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| feature | meaning | missing handling | categorical? | exists BEFORE the label window? |
|---|---|---|---|---|
| `impressions_prev30d` | sum of impressions, days −60…−31 | filter ≥ 70 (none below) | no | yes — strictly before |
| `impressions_last30d` | sum of impressions, days −30…0 | 0-filled in query; rows with GA4 off filtered on the flag | no | no — overlaps the label's window |
| `avg_position_30d` | mean `gsc_avg_position`, days −30…0 | only over days with rows; NA possible for sparse pages | no | no — overlaps the label's window |
| `clicks_30d` | sum of clicks, days −30…0 | 0-filled in query | no | no — it *is* the label's numerator |
| `search_volume` | static keyword demand | from `dim_content` | yes (looks numeric, categorical-like) | yes |
| `main_intent` | informational / transactional / commercial / navigational | from `dim_content`, `ANY_VALUE` | yes | yes |
| `content_type` | feedly article / keyword article | from `dim_content`, `ANY_VALUE` | yes | yes |

In [ ]:
print('content_type')
print(data_features.content_type.value_counts(), '\n')
print('main_intent')
print(data_features.main_intent.value_counts(), '\n')
print('missing per column')
print(data_features.isna().sum())

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Timeline first

Label: `ctr_label = clicks_30d / impressions_last30d`, defined over the **last 30 days** (2025-08-01 → 2025-08-31).

A feature may only use days **before** the label window opens. Any summary computed on the same calendar days as the label has already read the answer.

| feature | window | overlaps label window? | why it is dangerous |
|---|---|---|---|
| `impressions_prev30d` | −60…−31 | no | strictly before — a legal feature |
| `impressions_last30d` | −30…0 | **yes** | it is the label's *denominator* |
| `avg_position_30d` | −30…0 | **yes** | measured on the label's own days |
| `clicks_30d` | −30…0 | **yes** | it is the label's *numerator* |
| `search_volume`, `main_intent`, `content_type` | static | no | known before prediction |

The inventory is checked and the three tests below are run on the same feature vector.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

label = "ctr_label"

inventory = pd.DataFrame(
    [
        ("impressions_prev30d", "days -60..-31", "no",  "strictly before the label window"),
        ("impressions_last30d", "days -30..0",   "yes", "label's denominator"),
        ("avg_position_30d",    "days -30..0",   "yes", "same days as the label"),
        ("clicks_30d",          "days -30..0",   "yes", "label's numerator"),
        ("search_volume",       "static",        "no",  "external keyword demand"),
        ("main_intent",         "static",        "no",  "inferred from the query"),
        ("content_type",        "static",        "no",  "decided when the page is created"),
    ],
    columns=["feature", "window", "overlaps_label_window", "why"],
)
inventory

def prep(df, cols):
    """One-hot categoricals on the whole frame so train/test share every column."""
    X = pd.get_dummies(df[cols], dtype=int)
    y = df[label].copy()
    return X, y

def rf_rmse(X_tr, X_te, y_tr, y_te):
    """Fit an RF on the train fold, return test RMSE."""
    m = RandomForestRegressor(n_estimators=100, random_state=42)
    m.fit(X_tr, y_tr)
    return float(np.sqrt(mean_squared_error(y_te, m.predict(X_te))))

def base_rate_rmse(df):
    """Naive baseline: predict the mean CTR for every row."""
    y = df[label]
    return float(np.sqrt(mean_squared_error(y, np.full(len(y), y.mean()))))

### Demo A — feed the label's own numerator back in

Intentional violation (leak #1, label-derived feature). `clicks_30d` is the numerator of the label, so the model is handed the answer. RMSE collapsing toward ~0 is the confession.

In [ ]:
leaky_cols = ["main_intent", "content_type", "search_volume", "clicks_30d"]
X, y = prep(data_features, leaky_cols)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rmse_leaky = rf_rmse(X_tr, X_te, y_tr, y_te)
print(f"RMSE WITH clicks_30d (leaked): {rmse_leaky:.4f}")
print(f"RMSE base rate (predict the mean): {base_rate_rmse(data_features):.4f}")

### Demo B — drop clicks, keep the overlapping-window columns

`clicks_30d` is gone, but `impressions_last30d` (the label's **denominator**) and `avg_position_30d` are still measured over the very days the label describes. This is leak #2 (`future/overlapping` windows): a partial leak remains through the same-window columns, and the loss is still unrealistically small.

In [ ]:
overlap_cols = ["main_intent", "content_type", "search_volume",
                "impressions_last30d", "avg_position_30d"]
X, y = prep(data_features, overlap_cols)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rmse_overlap = rf_rmse(X_tr, X_te, y_tr, y_te)
print(f"RMSE WITHOUT clicks (overlapping windows left in): {rmse_overlap:.4f}")
print(f"RMSE base rate (predict the mean): {base_rate_rmse(data_features):.4f}")

### Demo C — honest set (every feature strictly before the label window)

Drop every same-window column. Only `impressions_prev30d` and static content attributes remain — the only features that actually exist at prediction time.

In [ ]:
honest_cols = ["main_intent", "content_type", "search_volume", "impressions_prev30d"]
X, y = prep(data_features, honest_cols)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rmse_random_honest = rf_rmse(X_tr, X_te, y_tr, y_te)
print(f"RMSE honest set, random split: {rmse_random_honest:.4f}")
print(f"RMSE base rate (predict the mean): {base_rate_rmse(data_features):.4f}")

### Demo D — grouped split by client + the base rate

A random split lets the model memorize per-client quirks. `GroupKFold` by `client_hash_id` answers the honest question: does it predict a client it **never saw**? The random-vs-grouped gap is a memoization finding. Every score sits next to the naive "predict the mean" baseline.

In [ ]:
X, y = prep(data_features, honest_cols)
groups = data_features["client_hash_id"].values
gkf = GroupKFold(n_splits=5)
fold_rmse = []
for tr, te in gkf.split(X, y, groups):
    fold_rmse.append(rf_rmse(X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]))

rmse_grouped = float(np.mean(fold_rmse))
print(f"RMSE honest set, grouped by client (mean of {len(fold_rmse)} folds): {rmse_grouped:.4f}")
print(f"RMSE honest set, random split: {rmse_random_honest:.4f}")
print(f"Gap (grouped - random): {rmse_grouped - rmse_random_honest:+.4f}  <- per-client memorization")
print(f"RMSE base rate (predict the mean): {base_rate_rmse(data_features):.4f}")

### Verdict

- **Demo A → B → C** is the three-way test: with the label-derived column the RMSE collapses; slicing it out but leaving same-window columns still looks too good; the honest, deployable number is the one that survives.
- `impressions_last30d` and `avg_position_30d` are excluded from the final vector — not because they are useless, but because they are measured on the same days as the label. Only `*_prev30`-style and static features stay.
- The grouped-split gap shows a random split overstates skill by letting the model memorize per-client patterns; the grouped number is the one to report.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded**
- `clicks_30d` — the label's numerator; feeding it in is the leak Demo A exposes.
- `impressions_last30d` — the label's denominator; same window as the label (Demo B).
- `avg_position_30d` — measured over the label's own days, so it is not knowable before the prediction moment (Demo B).
- `client_hash_id`, `content_hash_id` — pseudonymous IDs; grouping/splitting/joining only, never features.
- Rows where `ga4_data_available IS FALSE` — GA4 columns are zero-filled there; those zeros are not "no engagement", they were filtered out at the window level.

**Kept**
- `impressions_prev30d` — strictly before the label window.
- `search_volume`, `main_intent`, `content_type` — static, known before the prediction moment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.